# 98b — Combined Geode + nodal shot-gather wiggle plots

This notebook makes the actual combined visual product: one figure per common shot, with the stacked nodal gather and the corresponding Geode gather shown together.

It reads:
- `geode_nodal_common_shot_comparisons` from notebook 97
- `geode_events` from notebook 92

It writes/replaces only:
- `combined_wiggle_plot_files`
- `combined_wiggle_plot_errors`

In [1]:
from pathlib import Path
import sqlite3
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import read, UTCDateTime

PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"
CATALOG_DB.parent.mkdir(parents=True, exist_ok=True)

OUT_ROOT = PROJECT_ROOT / "combined_supergathers_v1" / "wiggle_plots"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

COMPONENT = "Z"
TARGET_SURVEYS = None     # e.g. ["T1_streamer_masw"]
MAX_PLOTS = None          # e.g. 10 for testing

TMIN_S = 0.0
TMAX_S = 0.8
CLIP_PERCENTILE = 99
TRACE_SCALE = 0.8

WRITE_TABLES = True

print("CATALOG_DB:", CATALOG_DB)
print("OUT_ROOT:", OUT_ROOT)

CATALOG_DB: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
OUT_ROOT: /Volumes/tachyon/LBSSP_DATA/combined_supergathers_v1/wiggle_plots


## Load comparison catalog

In [2]:
REQUIRED = ["geode_nodal_common_shot_comparisons", "geode_events"]
OWNED = ["combined_wiggle_plot_files", "combined_wiggle_plot_errors"]

if not CATALOG_DB.exists():
    raise FileNotFoundError(CATALOG_DB)

with sqlite3.connect(CATALOG_DB) as conn:
    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
        conn,
    )["name"].tolist()

missing = [t for t in REQUIRED if t not in tables]
if missing:
    raise RuntimeError(f"Missing required inputs: {missing}. Run 97 first.")

conn = sqlite3.connect(CATALOG_DB)
comparisons = pd.read_sql("SELECT * FROM geode_nodal_common_shot_comparisons", conn)
geode_events = pd.read_sql("SELECT * FROM geode_events", conn)

if TARGET_SURVEYS is not None:
    comparisons = comparisons[comparisons["survey"].astype(str).isin(TARGET_SURVEYS)].copy()

if MAX_PLOTS is not None:
    comparisons = comparisons.head(MAX_PLOTS).copy()

print("comparisons:", len(comparisons))
display(comparisons.groupby(["survey", "status"], dropna=False).size().reset_index(name="n"))
display(comparisons.head())

comparisons: 150


,survey,status,n
0,T1_1m_refraction,ready,34
1,T1_2m_refraction,ready,23
2,T1_streamer_masw,missing_file,66
3,T3_1m_refraction,ready,27


,comparison_id,stack_id,geode_event_id,survey,line,file_no,source_x_m,component,nodal_stack_mseed_path,nodal_stack_exists,geode_file_path,geode_file_exists,geode_read_format,geode_n_traces,status
0,CMP_NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,GEODE_T1_1M_REFRACTION_F3006,T1_1m_refraction,T1,3006,84.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,1,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,SEG2,72.0,ready
1,CMP_NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,GEODE_T1_1M_REFRACTION_F3008,T1_1m_refraction,T1,3008,88.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,1,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,SEG2,72.0,ready
2,CMP_NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,GEODE_T1_1M_REFRACTION_F3009,T1_1m_refraction,T1,3009,90.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,1,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,SEG2,72.0,ready
3,CMP_NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,GEODE_T1_1M_REFRACTION_F3011,T1_1m_refraction,T1,3011,94.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,1,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,SEG2,72.0,ready
4,CMP_NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,GEODE_T1_1M_REFRACTION_F3012,T1_1m_refraction,T1,3012,96.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,1,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,SEG2,72.0,ready


## Diagnostic: paths and association logic

In [3]:

# Association chain:
#   94: geode_event_id <-> nodal_event_id candidates
#   95: geode_event_id -> stack_id for stacked nodal gather
#   97: stack_id + geode_event_id -> comparison row with nodal_stack_mseed_path and geode_file_path
#
# 98b reads:
#   nodal_stack_mseed_path from geode_nodal_common_shot_comparisons
#   geode_file_path from geode_nodal_common_shot_comparisons / geode_events

path_cols = [
    "comparison_id", "stack_id", "geode_event_id", "survey", "file_no",
    "source_x_m", "nodal_stack_mseed_path", "geode_file_path",
    "nodal_stack_exists", "geode_file_exists", "status"
]
display(comparisons[[c for c in path_cols if c in comparisons.columns]].head(20))

print("Geode file exists counts:")
if "geode_file_exists" in comparisons.columns:
    display(comparisons["geode_file_exists"].value_counts(dropna=False))

print("Nodal stack exists counts:")
if "nodal_stack_exists" in comparisons.columns:
    display(comparisons["nodal_stack_exists"].value_counts(dropna=False))


,comparison_id,stack_id,geode_event_id,survey,file_no,source_x_m,nodal_stack_mseed_path,geode_file_path,nodal_stack_exists,geode_file_exists,status
0,CMP_NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,GEODE_T1_1M_REFRACTION_F3006,T1_1m_refraction,3006,84.5,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,1,ready
1,CMP_NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,GEODE_T1_1M_REFRACTION_F3008,T1_1m_refraction,3008,88.5,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,1,ready
2,CMP_NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,GEODE_T1_1M_REFRACTION_F3009,T1_1m_refraction,3009,90.5,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,1,ready
3,CMP_NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,GEODE_T1_1M_REFRACTION_F3011,T1_1m_refraction,3011,94.5,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,1,ready
4,CMP_NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,GEODE_T1_1M_REFRACTION_F3012,T1_1m_refraction,3012,96.5,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,1,ready
5,CMP_NODALSTACK_T1_T1_1m_refraction_F3013_x0098.5m,NODALSTACK_T1_T1_1m_refraction_F3013_x0098.5m,GEODE_T1_1M_REFRACTION_F3013,T1_1m_refraction,3013,98.5,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,1,ready
6,CMP_NODALSTACK_T1_T1_1m_refraction_F3016_x0102.5m,NODALSTACK_T1_T1_1m_refraction_F3016_x0102.5m,GEODE_T1_1M_REFRACTION_F3016,T1_1m_refraction,3016,102.5,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,1,ready
7,CMP_NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,GEODE_T1_1M_REFRACTION_F3018,T1_1m_refraction,3018,106.5,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,1,ready
8,CMP_NODALSTACK_T1_T1_1m_refraction_F3019_x0108.5m,NODALSTACK_T1_T1_1m_refraction_F3019_x0108.5m,GEODE_T1_1M_REFRACTION_F3019,T1_1m_refraction,3019,108.5,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,1,ready
9,CMP_NODALSTACK_T1_T1_1m_refraction_F3020_x0110.5m,NODALSTACK_T1_T1_1m_refraction_F3020_x0110.5m,GEODE_T1_1M_REFRACTION_F3020,T1_1m_refraction,3020,110.5,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,1,ready


Geode file exists counts:


geode_file_exists
1    84
0    66
Name: count, dtype: int64

Nodal stack exists counts:


nodal_stack_exists
1    150
Name: count, dtype: int64

## Helper functions

In [4]:
def safe_name(s):
    s = str(s)
    for ch in [" ", "/", "\\", ":", ";", ",", "(", ")", "[", "]"]:
        s = s.replace(ch, "_")
    return s

def get_time_vector(tr):
    return tr.stats.starttime - UTCDateTime(0) + np.arange(tr.stats.npts) * tr.stats.delta

def infer_geode_receiver_xs(geode_row, n_traces):
    first = pd.to_numeric(geode_row.get("receiver_first_m"), errors="coerce")
    last = pd.to_numeric(geode_row.get("receiver_last_m"), errors="coerce")
    spacing = pd.to_numeric(geode_row.get("receiver_spacing_m"), errors="coerce")

    if np.isfinite(first) and np.isfinite(spacing) and spacing != 0:
        return (first + np.arange(n_traces) * spacing).astype(float), "receiver_first_m + receiver_spacing_m"
    if np.isfinite(first) and np.isfinite(last) and n_traces > 1:
        return np.linspace(first, last, n_traces).astype(float), "linspace(receiver_first_m, receiver_last_m)"
    return np.arange(n_traces, dtype=float), "trace_index_fallback"

def attach_geode_geometry(st, geode_row):
    xs, method = infer_geode_receiver_xs(geode_row, len(st))
    out = st.copy()
    for tr, x in zip(out, xs):
        tr.stats.receiver_x_m = float(x)
    return out, method

def select_component_for_plot(st, component=COMPONENT):
    stc = st.select(channel=f"*{component}").copy()
    return stc if len(stc) else st.copy()

def prepare_trace_arrays(st, tmin=TMIN_S, tmax=TMAX_S, origin=None):
    """
    Convert a stream to relative-time wiggle arrays.

    Nodal stacked traces from 95 are written near UTCDateTime(0), so use origin=UTCDateTime(0).
    Raw Geode files usually have real absolute start times, so use origin=min(trace.starttime).
    """
    rows = []
    if len(st) == 0:
        return rows

    if origin is None:
        origin = UTCDateTime(0)

    for i, tr in enumerate(st):
        x = float(getattr(tr.stats, "receiver_x_m", i))
        dt = float(tr.stats.delta)
        t = (tr.stats.starttime - origin) + np.arange(tr.stats.npts) * dt
        grid = np.arange(tmin, tmax + 0.5 * dt, dt)
        y = np.interp(grid, t, tr.data.astype(float), left=np.nan, right=np.nan)
        if np.isfinite(y).sum() < 5:
            continue
        y = y - np.nanmedian(y)
        rows.append((x, grid, y, tr.id))
    return sorted(rows, key=lambda r: r[0])

def plot_wiggles_on_axis(ax, rows, title, source_x_m=None):
    if not rows:
        ax.text(0.5, 0.5, "No readable traces", transform=ax.transAxes, ha="center", va="center")
        ax.set_title(title)
        return

    all_y = np.concatenate([r[2][np.isfinite(r[2])] for r in rows if np.isfinite(r[2]).any()])
    clip = np.nanpercentile(np.abs(all_y), CLIP_PERCENTILE) if len(all_y) else 1.0
    if not np.isfinite(clip) or clip <= 0:
        clip = 1.0

    xs = np.array([r[0] for r in rows], dtype=float)
    ux = np.sort(np.unique(xs[np.isfinite(xs)]))
    dx = np.nanmedian(np.diff(ux)) if len(ux) > 1 else 1.0
    if not np.isfinite(dx) or dx <= 0:
        dx = 1.0

    for x, t, y, _ in rows:
        yy = np.clip(y / clip, -1, 1) * dx * TRACE_SCALE
        ax.plot(x + yy, t, linewidth=0.6)
        ax.fill_betweenx(t, x, x + np.maximum(yy, 0), alpha=0.20)

    if source_x_m is not None and np.isfinite(source_x_m):
        ax.axvline(float(source_x_m), linestyle="--", linewidth=1.0, label=f"source x={source_x_m:.1f} m")
        ax.legend(loc="best", fontsize=8)

    ax.invert_yaxis()
    ax.set_ylabel("Time (s)")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)

def make_combined_plot(row, geode_row, out_png):
    source_x = pd.to_numeric(row.get("source_x_m"), errors="coerce")
    nodal_path = row.get("nodal_stack_mseed_path")
    geode_path = row.get("geode_file_path")

    nodal_rows = []
    geode_rows = []
    geode_geom_method = None
    geode_read_error = None

    if nodal_path and Path(str(nodal_path)).exists():
        st_nodal = read(str(nodal_path))
        st_nodal = select_component_for_plot(st_nodal, COMPONENT)
        nodal_rows = prepare_trace_arrays(st_nodal, origin=UTCDateTime(0))

    if geode_path and Path(str(geode_path)).exists():
        try:
            st_geode = read(str(geode_path))
            st_geode = select_component_for_plot(st_geode, COMPONENT)
            st_geode, geode_geom_method = attach_geode_geometry(st_geode, geode_row)
            geode_origin = min(tr.stats.starttime for tr in st_geode) if len(st_geode) else UTCDateTime(0)
            geode_rows = prepare_trace_arrays(st_geode, origin=geode_origin)
        except Exception as e:
            geode_read_error = repr(e)

    fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=False)

    plot_wiggles_on_axis(
        axes[0],
        nodal_rows,
        title=f"Nodal stacked gather | {row.get('survey')} F{row.get('file_no')} | x={source_x:.1f} m",
        source_x_m=source_x,
    )

    geode_title = f"Geode gather | geometry: {geode_geom_method or 'not available'}"
    if geode_read_error:
        geode_title += f" | read error: {geode_read_error[:80]}"
    plot_wiggles_on_axis(axes[1], geode_rows, title=geode_title, source_x_m=source_x)

    axes[1].set_xlabel("Receiver x (m), or fallback trace index")
    fig.suptitle(
        f"Combined nodal + Geode shot gather comparison\\n"
        f"{row.get('line')} {row.get('survey')} | {row.get('geode_event_id')}",
        y=0.995,
    )
    plt.tight_layout()

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=180)
    plt.close(fig)

    return {
        "nodal_n_traces_plotted": len(nodal_rows),
        "geode_n_traces_plotted": len(geode_rows),
        "geode_geometry_method": geode_geom_method,
        "geode_read_error": geode_read_error,
        "geode_file_path_used": str(geode_path),
        "nodal_file_path_used": str(nodal_path),
    }

## Make combined wiggle plots

In [5]:
file_rows = []
error_rows = []

for i, row in comparisons.iterrows():
    comparison_id = row["comparison_id"]
    print(f"[{i+1}/{len(comparisons)}] {comparison_id}")

    ge = geode_events[geode_events["geode_event_id"].astype(str).eq(str(row["geode_event_id"]))]
    if ge.empty:
        error_rows.append({
            "comparison_id": comparison_id,
            "stage": "lookup_geode_event",
            "error": "geode_event_id not found in geode_events",
        })
        continue

    try:
        out_png = (
            OUT_ROOT
            / safe_name(row.get("line"))
            / safe_name(row.get("survey"))
            / f"{safe_name(comparison_id)}_combined_wiggle.png"
        )
        info = make_combined_plot(row, ge.iloc[0], out_png)

        file_rows.append({
            "comparison_id": comparison_id,
            "stack_id": row.get("stack_id"),
            "geode_event_id": row.get("geode_event_id"),
            "survey": row.get("survey"),
            "line": row.get("line"),
            "file_no": row.get("file_no"),
            "source_x_m": row.get("source_x_m"),
            "component": COMPONENT,
            "file_type": "png_combined_geode_nodal_wiggle",
            "file_path": str(out_png),
            **info,
        })

        if info.get("geode_read_error"):
            error_rows.append({
                "comparison_id": comparison_id,
                "stage": "read_geode",
                "file_path": row.get("geode_file_path"),
                "error": info.get("geode_read_error"),
            })

    except Exception as e:
        error_rows.append({
            "comparison_id": comparison_id,
            "stage": "make_combined_plot",
            "error": repr(e),
            "traceback": traceback.format_exc(),
        })

combined_wiggle_plot_files = pd.DataFrame(file_rows)
combined_wiggle_plot_errors = pd.DataFrame(error_rows)

print("Plots:", len(combined_wiggle_plot_files))
print("Errors:", len(combined_wiggle_plot_errors))
display(combined_wiggle_plot_files.head())
display(combined_wiggle_plot_errors.head())

[1/150] CMP_NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m


/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:365: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influence e.g. trace start times.
  warnings.warn(WARNING_HEADER)


[2/150] CMP_NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m


/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:365: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influence e.g. trace start times.
  warnings.warn(WARNING_HEADER)


[3/150] CMP_NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m


...

[150/150] CMP_NODALSTACK_T3_T3_1m_refraction_F4036_x0069.5m


/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:365: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influence e.g. trace start times.
  warnings.warn(WARNING_HEADER)


Plots: 150
Errors: 0


,comparison_id,stack_id,geode_event_id,survey,line,file_no,source_x_m,component,file_type,file_path,nodal_n_traces_plotted,geode_n_traces_plotted,geode_geometry_method,geode_read_error,geode_file_path_used,nodal_file_path_used
0,CMP_NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,GEODE_T1_1M_REFRACTION_F3006,T1_1m_refraction,T1,3006,84.5,Z,png_combined_geode_nodal_wiggle,/Volumes/tachyon/LBSSP_DATA/combined_supergath...,34,72,receiver_first_m + receiver_spacing_m,None,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...
1,CMP_NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,GEODE_T1_1M_REFRACTION_F3008,T1_1m_refraction,T1,3008,88.5,Z,png_combined_geode_nodal_wiggle,/Volumes/tachyon/LBSSP_DATA/combined_supergath...,34,72,receiver_first_m + receiver_spacing_m,None,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...
2,CMP_NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,GEODE_T1_1M_REFRACTION_F3009,T1_1m_refraction,T1,3009,90.5,Z,png_combined_geode_nodal_wiggle,/Volumes/tachyon/LBSSP_DATA/combined_supergath...,34,72,receiver_first_m + receiver_spacing_m,None,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...
3,CMP_NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,GEODE_T1_1M_REFRACTION_F3011,T1_1m_refraction,T1,3011,94.5,Z,png_combined_geode_nodal_wiggle,/Volumes/tachyon/LBSSP_DATA/combined_supergath...,34,72,receiver_first_m + receiver_spacing_m,None,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...
4,CMP_NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,GEODE_T1_1M_REFRACTION_F3012,T1_1m_refraction,T1,3012,96.5,Z,png_combined_geode_nodal_wiggle,/Volumes/tachyon/LBSSP_DATA/combined_supergath...,34,72,receiver_first_m + receiver_spacing_m,None,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...


""


## Write notebook-owned tables and CSV exports

In [6]:
if combined_wiggle_plot_files is None or len(combined_wiggle_plot_files.columns) == 0:
    combined_wiggle_plot_files = pd.DataFrame(
        columns=["comparison_id", "stack_id", "geode_event_id", "file_type", "file_path"]
    )

if combined_wiggle_plot_errors is None or len(combined_wiggle_plot_errors.columns) == 0:
    combined_wiggle_plot_errors = pd.DataFrame(columns=["comparison_id", "stage", "error"])

combined_wiggle_plot_files.to_csv(OUT_ROOT / "combined_wiggle_plot_files.csv", index=False)
combined_wiggle_plot_errors.to_csv(OUT_ROOT / "combined_wiggle_plot_errors.csv", index=False)

if WRITE_TABLES:
    with sqlite3.connect(CATALOG_DB) as conn:
        for t in OWNED:
            conn.execute(f'DROP TABLE IF EXISTS "{t}"')

        combined_wiggle_plot_files.to_sql("combined_wiggle_plot_files", conn, if_exists="fail", index=False)
        combined_wiggle_plot_errors.to_sql("combined_wiggle_plot_errors", conn, if_exists="fail", index=False)
        conn.commit()

print("Wrote tables:", OWNED)
print("CSV/file outputs:", OUT_ROOT)

if len(combined_wiggle_plot_files):
    display(
        combined_wiggle_plot_files
        .groupby(["survey", "file_type"], dropna=False)
        .size()
        .reset_index(name="n")
    )

Wrote tables: ['combined_wiggle_plot_files', 'combined_wiggle_plot_errors']
CSV/file outputs: /Volumes/tachyon/LBSSP_DATA/combined_supergathers_v1/wiggle_plots


,survey,file_type,n
0,T1_1m_refraction,png_combined_geode_nodal_wiggle,34
1,T1_2m_refraction,png_combined_geode_nodal_wiggle,23
2,T1_streamer_masw,png_combined_geode_nodal_wiggle,66
3,T3_1m_refraction,png_combined_geode_nodal_wiggle,27
